# GLM-OCR vs PaddleOCR-VL-1.5 Benchmark

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/szymonkulpinski/colab_notebooks/blob/main/benchmark_glm_ocr_vs_paddleocr_vl.ipynb)

---

This notebook benchmarks **GLM-OCR** (0.9B) against **PaddleOCR-VL-1.5** (0.9B) across 8 diverse OCR datasets, and compares both against a fine-tuned **SmolVLM2** specialist on jersey number recognition.

| # | Dataset | Task |
|---|---------|------|
| 1 | CAPTCHA | Distorted alphanumeric text recognition |
| 2 | LaTeX equations | Mathematical formula extraction |
| 3 | Receipts (3 fields) | Structured field extraction |
| 4 | Date stamps | Date extraction (YYYY-MM-DD) |
| 5 | Basketball jersey numbers | Short numeric OCR |
| 6 | Container serial numbers | Alphanumeric serial codes |
| 7 | Tire codes | Embossed alphanumeric codes |
| 8 | License plates | Vehicle plate recognition |

**Metrics**: Exact Match Accuracy, Character Error Rate (CER), Normalized Edit Distance (NED)  
**See** `ocr_metrics_research.md` for metric rationale.

## 1. Install Dependencies

In [1]:
%%capture
!pip install git+https://github.com/huggingface/transformers.git accelerate torch torchvision pillow -q
!pip install roboflow supervision -q
!pip install Levenshtein matplotlib pandas -q

## 2. Configuration & Secrets

In [ ]:
import os
from google.colab import userdata

ROBOFLOW_API_KEY = userdata.get('ROBOFLOW_API_KEY')
HF_TOKEN = userdata.get('HF_TOKEN')  # required for SmolVLM2 (Section 11); free at huggingface.co

os.environ['ROBOFLOW_API_KEY'] = ROBOFLOW_API_KEY
os.environ['HF_TOKEN'] = HF_TOKEN  # needed by SmolVLM2 model download

# Dataset slugs from Roboflow (workspace/project/version)
# jsonl = Roboflow text-image-pairs (two-step: jsonl for images, openai for labels)
# coco  = object-detection COCO format (tire: per-character bboxes, sorted left→right)
DATASETS = {
    "captcha":   {"url": "https://universe.roboflow.com/roboflow-jvuqo/captcha-ocr-g5ol8/dataset/4",   "format": "jsonl"},
    "latex":     {"url": "https://universe.roboflow.com/roboflow-jvuqo/latex-ocr-subset/3",            "format": "jsonl"},
    "datestamp": {"url": "https://universe.roboflow.com/roboflow-jvuqo/date-stamp-extraction/1",       "format": "jsonl"},
    "jersey":    {"url": "https://universe.roboflow.com/roboflow-jvuqo/basketball-jersey-numbers-ocr/7", "format": "jsonl"},
    "tire":      {"url": "https://universe.roboflow.com/bence-toth-h5oec/tire-serial/dataset/1",       "format": "coco"},
}

# Prompts for each task (same for both models).
PROMPTS = {
    "captcha":   "What text is shown in this CAPTCHA image? Return only the exact characters, no explanation.",
    "latex":     "Extract the mathematical formula from this image as LaTeX. Return only the LaTeX code, no explanation.",
    "datestamp": "What date is shown in this image? Return only the date in YYYY-MM-DD format, no explanation.",
    "jersey":    "What number is on this jersey? Return only the number, no explanation.",
    "tire":      "What code is printed on this tire? Return only the code, no explanation.",
}

# PaddleOCR-VL-1.5 uses short task-prefix prompts that match its training.
PADDLE_PROMPTS = {
    "captcha":   "What text is shown in this CAPTCHA image? Return only the exact characters, no explanation.",
    "latex":     "Formula Recognition:",
    "datestamp": "OCR:",
    "jersey":    "OCR:",
    "tire":      "OCR:",
}

N_SAMPLES = None  # evaluate on full dataset
BATCH_SIZE = 8    # tuned for L4 (24 GB); reduce to 4 on T4, or 1 if OOM


## 3. Metric Utilities

In [ ]:
import Levenshtein
import json
import re

def normalise(text: str) -> str:
    """Lowercase, strip whitespace."""
    return text.strip().lower()

def exact_match(pred: str, ref: str) -> int:
    """Return 1 if prediction matches reference after normalisation, else 0."""
    return int(normalise(pred) == normalise(ref))

def cer(pred: str, ref: str) -> float:
    """Character Error Rate = edit_distance / len(reference)."""
    ref_n = normalise(ref)
    if len(ref_n) == 0:
        return 0.0 if len(normalise(pred)) == 0 else 1.0
    return Levenshtein.distance(normalise(pred), ref_n) / len(ref_n)

def ned(pred: str, ref: str) -> float:
    """Normalized Edit Distance = edit_distance / max(len(pred), len(ref))."""
    p, r = normalise(pred), normalise(ref)
    denom = max(len(p), len(r))
    if denom == 0:
        return 0.0
    return Levenshtein.distance(p, r) / denom

def evaluate(predictions: list[str], references: list[str]) -> dict:
    """Compute aggregate metrics for plain-text predictions vs references."""
    assert len(predictions) == len(references)
    ems = [exact_match(p, r) for p, r in zip(predictions, references)]
    cers = [cer(p, r) for p, r in zip(predictions, references)]
    neds = [ned(p, r) for p, r in zip(predictions, references)]
    return {
        "exact_match": sum(ems) / len(ems),
        "cer": sum(cers) / len(cers),
        "ned": sum(neds) / len(neds),
        "n": len(ems),
    }


## 4. Dataset Loading

In [ ]:
import os
from pathlib import Path
from PIL import Image
from roboflow import download_dataset

IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png", ".bmp", ".webp")


def _load_jsonl(location: str, key: str) -> list[dict]:
    """
    Load a Roboflow text-image-pairs dataset from a jsonl download.

    Each split directory contains:
      - images (*.jpg / *.png)
      - annotations.jsonl  — one JSON line per image:
            {"image": "filename.rf.hash.jpg", "prefix": "...", "suffix": "label"}

    The 'image' field is a bare filename matching the locally downloaded file.
    The 'suffix' field is the ground-truth label.
    """
    root = Path(location)
    samples = []

    for ann_file in sorted(root.rglob("annotations.jsonl")):
        split_dir = ann_file.parent
        content = ann_file.read_text(errors="replace").strip()
        if not content:
            continue

        for line in content.splitlines():
            if not line.strip():
                continue
            try:
                item = json.loads(line)
            except json.JSONDecodeError:
                continue

            filename = item.get("image", "")
            label = item.get("suffix", "").strip()
            if not filename or not label:
                continue

            # The 'image' field is a bare filename; images live in the split dir
            img_path = split_dir / filename
            if not img_path.exists():
                continue

            samples.append({"image_path": str(img_path), "label": label})

    print(f"[{key}] {len(samples)} samples loaded across all splits")
    if samples:
        print(f"[{key}] sample label: {samples[0]['label']!r}")
    elif not any(True for _ in root.rglob("annotations.jsonl")):
        print(f"[{key}] WARNING: no annotations.jsonl found under {root}")
    else:
        print(f"[{key}] WARNING: annotations.jsonl found but all entries skipped (empty labels or missing images)")
    return samples


def _load_coco(location: str, key: str) -> list[dict]:
    """Load ALL image+label pairs from a Roboflow COCO download.

    Merges all _annotations.coco.json files found recursively so every
    split is included. Tries annotation fields in order:
    caption > text > value > attributes.text > attributes.value > category name.

    Annotations are sorted left-to-right by bounding box center_x, which
    reconstructs the correct character order for per-character detection datasets
    like tire serial codes.
    """
    root = Path(location)
    coco_files = sorted(root.rglob("_annotations.coco.json"))
    if not coco_files:
        print(f"[{key}] WARNING: no _annotations.coco.json found under {root}")
        for p in sorted(root.rglob("*"))[:30]:
            print(f"  {p.relative_to(root)}")
        return []

    samples = []
    for coco_file in coco_files:
        split_dir = coco_file.parent
        with open(coco_file) as f:
            coco = json.load(f)

        cat_map = {c["id"]: c["name"] for c in coco.get("categories", [])}
        ann_by_img = {}
        for ann in coco.get("annotations", []):
            iid = ann["image_id"]
            text = (
                ann.get("caption") or
                ann.get("text") or
                ann.get("value") or
                (ann.get("attributes") or {}).get("text") or
                (ann.get("attributes") or {}).get("value") or
                cat_map.get(ann.get("category_id"), "")
            )
            # Store (center_x, text) so we can sort left→right for
            # object-detection datasets where each bbox is one character.
            bbox = ann.get("bbox")  # COCO format: [x, y, w, h]
            center_x = (bbox[0] + bbox[2] / 2) if bbox else float("inf")
            ann_by_img.setdefault(iid, []).append((center_x, str(text).strip()))

        for img in coco.get("images", []):
            img_path = split_dir / img["file_name"]
            if not img_path.exists():
                continue
            entries = sorted(ann_by_img.get(img["id"], []), key=lambda e: e[0])
            label = "".join(t for _, t in entries if t).strip()
            samples.append({"image_path": str(img_path), "label": label})

    print(f"[{key}] {len(samples)} samples total across all splits (coco)")
    if samples and samples[0]["label"]:
        print(f"[{key}] sample label: {samples[0]['label']!r}")
    elif samples:
        print(f"[{key}] WARNING: labels appear empty — check COCO annotation fields")
    return samples


def load_dataset(key: str, n: int = None) -> list[dict]:
    """Download a Roboflow dataset and return [{image_path, label}] dicts."""
    cfg = DATASETS[key]
    fmt = cfg["format"]
    location = f"/tmp/rf_{key}"

    dataset = download_dataset(cfg["url"], fmt, location=location)

    if fmt == "jsonl":
        samples = _load_jsonl(dataset.location, key)
    else:  # coco
        samples = _load_coco(dataset.location, key)

    if not samples:
        print(f"[{key}] SKIPPING — no labelled samples (dataset may have no ground-truth annotations)")
        return []

    if n is not None:
        samples = samples[:n]
    return samples


## 5. Load Models

## Check GPU

In [5]:
!nvidia-smi

Thu Mar 26 11:25:43 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [6]:
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
    DTYPE = torch.float16
else:
    DEVICE = torch.device("cpu")
    DTYPE = torch.float32

print(f"Device: {DEVICE}  |  dtype: {DTYPE}")

Device: cuda  |  dtype: torch.bfloat16


In [7]:
# --- GLM-OCR ---
print("Loading GLM-OCR...")
GLM_MODEL_ID = "zai-org/GLM-OCR"

glm_processor = AutoProcessor.from_pretrained(GLM_MODEL_ID)
if DEVICE.type == "cuda":
    glm_model = AutoModelForImageTextToText.from_pretrained(
        GLM_MODEL_ID,
        torch_dtype=DTYPE,
        device_map="auto",
    )
else:
    glm_model = AutoModelForImageTextToText.from_pretrained(
        GLM_MODEL_ID,
        torch_dtype=DTYPE,
    ).to(DEVICE)
glm_model = glm_model.eval()
print("GLM-OCR loaded.")

Loading GLM-OCR...


OSError: THUDM/glm-ocr is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `hf auth login` or by passing `token=<your_token>`

In [ ]:
# --- PaddleOCR-VL-1.5 ---
print("Loading PaddleOCR-VL-1.5...")
PADDLE_MODEL_ID = "PaddlePaddle/PaddleOCR-VL-1.5"

paddle_processor = AutoProcessor.from_pretrained(PADDLE_MODEL_ID)
paddle_model = AutoModelForImageTextToText.from_pretrained(
    PADDLE_MODEL_ID,
    torch_dtype=DTYPE,
).to(DEVICE).eval()
print("PaddleOCR-VL-1.5 loaded.")

## 6. Inference Functions

In [ ]:
from tqdm.auto import tqdm as _tqdm

# --- GLM-OCR: sequential (apply_chat_template handles image+text together) ---
def run_glm_ocr_all(images: list, prompt: str) -> list:
    """
    Run GLM-OCR on a list of images sequentially.
    GLM-OCR's processor bundles image preprocessing inside apply_chat_template
    (tokenize=True, return_dict=True), so true batching requires manual
    collation; sequential is simpler and equally correct.
    token_type_ids must be removed — GLM-OCR's generate() does not accept them.
    """
    preds = []
    for image in _tqdm(images, desc="GLM-OCR", leave=False):
        messages = [{"role": "user", "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": prompt},
        ]}]
        inputs = glm_processor.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_dict=True,
            return_tensors="pt",
        ).to(glm_model.device)
        inputs.pop("token_type_ids", None)
        with torch.inference_mode():
            output_ids = glm_model.generate(**inputs, max_new_tokens=2048, do_sample=False)
        generated = output_ids[0, inputs["input_ids"].shape[1]:]
        preds.append(glm_processor.decode(generated, skip_special_tokens=True).strip())
    return preds


# --- PaddleOCR-VL-1.5: true batching (left-pad, generate, decode) -----------
def _paddle_batch(images: list, prompt: str) -> list:
    texts = []
    for img in images:
        messages = [{"role": "user", "content": [
            {"type": "image", "image": img},  # image passed explicitly
            {"type": "text",  "text":  prompt},
        ]}]
        texts.append(paddle_processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        ))
    paddle_processor.tokenizer.padding_side = "left"
    inputs = paddle_processor(
        text=texts, images=images, return_tensors="pt", padding=True
    )
    inputs = {k: (v.to(DEVICE) if isinstance(v, torch.Tensor) else v) for k, v in inputs.items()}
    with torch.inference_mode():
        output_ids = paddle_model.generate(**inputs, max_new_tokens=2048, do_sample=False)
    generated = output_ids[:, inputs["input_ids"].shape[1]:]
    return [paddle_processor.decode(g, skip_special_tokens=True).strip() for g in generated]


def run_paddle_ocr_all(images: list, prompt: str, batch_size: int) -> list:
    """Run PaddleOCR-VL-1.5 in batches with OOM fallback to batch_size=1."""
    preds = []
    i = 0
    pbar = _tqdm(total=len(images), desc="PaddleOCR", leave=False)
    while i < len(images):
        batch = images[i:i + batch_size]
        try:
            preds.extend(_paddle_batch(batch, prompt))
            i += batch_size
            pbar.update(len(batch))
        except torch.cuda.OutOfMemoryError:
            if batch_size == 1:
                raise
            print(f"OOM at batch_size={batch_size}, retrying with batch_size=1")
            batch_size = 1
            torch.cuda.empty_cache()
    pbar.close()
    return preds

## 7. Run Benchmark

This cell runs both models on all 8 datasets and stores per-sample predictions.

In [ ]:
results = {}

for dataset_key in DATASETS:
    print(f"\n{'='*50}")
    print(f"Dataset: {dataset_key}")
    print(f"{'='*50}")

    samples = load_dataset(dataset_key, n=N_SAMPLES)
    if not samples:
        continue  # no ground-truth labels — skip

    image_paths   = [s["image_path"] for s in samples]
    labels        = [s["label"]      for s in samples]
    prompt        = PROMPTS[dataset_key]
    paddle_prompt = PADDLE_PROMPTS[dataset_key]

    print(f"Loading {len(samples)} images...")
    images = [Image.open(p).convert("RGB") for p in image_paths]

    print(f"Running GLM-OCR...")
    glm_preds = run_glm_ocr_all(images, prompt)

    print(f"Running PaddleOCR-VL-1.5 (batch_size={BATCH_SIZE})...")
    paddle_preds = run_paddle_ocr_all(images, paddle_prompt, BATCH_SIZE)

    glm_metrics    = evaluate(glm_preds,    [str(l) for l in labels])
    paddle_metrics = evaluate(paddle_preds, [str(l) for l in labels])
    print(f"  GLM-OCR     | EM={glm_metrics['exact_match']:.3f}  CER={glm_metrics['cer']:.3f}  NED={glm_metrics['ned']:.3f}")
    print(f"  PaddleOCR   | EM={paddle_metrics['exact_match']:.3f}  CER={paddle_metrics['cer']:.3f}  NED={paddle_metrics['ned']:.3f}")

    results[dataset_key] = {
        "image_paths": image_paths,
        "glm":    {**glm_metrics,    "predictions": glm_preds},
        "paddle": {**paddle_metrics, "predictions": paddle_preds},
        "labels": labels,
    }

print("\nBenchmark complete!")


In [ ]:
# Mount Google Drive and save results — survives session restarts and crashes
from google.colab import drive
from datetime import datetime

drive.mount("/content/drive", force_remount=False)

import json as _json

_ts = datetime.now().strftime("%Y%m%d_%H%M%S")
RESULTS_PATH = f"/content/drive/MyDrive/ocr_benchmark/benchmark_results_{_ts}.json"
os.makedirs(os.path.dirname(RESULTS_PATH), exist_ok=True)

with open(RESULTS_PATH, "w") as _f:
    _json.dump(results, _f, indent=2, ensure_ascii=False)

size_kb = os.path.getsize(RESULTS_PATH) / 1024
print(f"Results saved → {RESULTS_PATH}  ({size_kb:.1f} KB)")
print(f"Datasets: {list(results.keys())}")


In [ ]:
# ── OPTIONAL: restore a previous run from Google Drive ──────────────────────
# Skip this cell if `results` is already in memory from the benchmark above.
# Run it instead of Section 7 when you only want to re-plot saved results.
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import json as _json

RESULTS_PATH = "/content/drive/MyDrive/ocr_benchmark/benchmark_results.json"

with open(RESULTS_PATH) as _f:
    results = _json.load(_f)

print(f"Loaded results from {RESULTS_PATH}")
for ds, res in results.items():
    models = [k for k in res if k not in ("labels", "image_paths")]
    n = len(res["labels"])
    print(f"  {ds:12s}  {n:4d} samples  models={models}")

## 8. Results Summary Table

In [ ]:
import pandas as pd

rows = []
for ds, res in results.items():
    for model_name, model_key in [("GLM-OCR", "glm"), ("PaddleOCR-VL-1.5", "paddle")]:
        m = res[model_key]
        rows.append({
            "Dataset": ds,
            "Model": model_name,
            "Exact Match": round(m["exact_match"], 4),
            "CER": round(m["cer"], 4),
            "NED": round(m["ned"], 4),
            "N": m["n"],
        })

df = pd.DataFrame(rows)
print("=== Full Results ===")
print(df.to_string(index=False))

print("\n=== Exact Match pivot ===")
df_pivot = df[["Dataset", "Model", "Exact Match"]].pivot(
    index="Dataset", columns="Model", values="Exact Match"
)
df_pivot["Winner"] = df_pivot.apply(
    lambda r: "GLM-OCR" if r["GLM-OCR"] > r["PaddleOCR-VL-1.5"]
              else ("PaddleOCR-VL-1.5" if r["PaddleOCR-VL-1.5"] > r["GLM-OCR"] else "Tie"),
    axis=1
)
print(df_pivot.to_string())


## 9. Visualisations

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np

DATASET_LABELS = {
    "captcha":   "CAPTCHA",
    "latex":     "LaTeX",
    "datestamp": "Date\nStamp",
    "jersey":    "Jersey\nNumber",
    "tire":      "Tire Code",
}

datasets = list(results.keys())
x = np.arange(len(datasets))
width = 0.35
xtick_labels = [DATASET_LABELS.get(d, d) for d in datasets]

COLORS = {"GLM-OCR": "#4C72B0", "PaddleOCR-VL-1.5": "#DD8452"}


In [ ]:
# --- Plot 1: Exact Match Accuracy (grouped bar) ---
fig, ax = plt.subplots(figsize=(12, 5))

glm_em = [results[d]["glm"]["exact_match"] for d in datasets]
paddle_em = [results[d]["paddle"]["exact_match"] for d in datasets]

bars1 = ax.bar(x - width/2, glm_em, width, label="GLM-OCR", color=COLORS["GLM-OCR"], alpha=0.85)
bars2 = ax.bar(x + width/2, paddle_em, width, label="PaddleOCR-VL-1.5", color=COLORS["PaddleOCR-VL-1.5"], alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(xtick_labels, fontsize=10)
ax.set_ylabel("Exact Match Accuracy", fontsize=12)
ax.set_title("Exact Match Accuracy by Dataset", fontsize=14, fontweight="bold")
ax.set_ylim(0, 1.12)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
ax.legend(fontsize=11)
ax.bar_label(bars1, fmt="%.0f%%", label_type="edge", fontsize=8, padding=2,
             labels=[f"{v*100:.0f}%" for v in glm_em])
ax.bar_label(bars2, fmt="%.0f%%", label_type="edge", fontsize=8, padding=2,
             labels=[f"{v*100:.0f}%" for v in paddle_em])
ax.grid(axis="y", alpha=0.3)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig("plot_exact_match.png", dpi=150)
plt.show()

In [ ]:
# --- Plot 2: Character Error Rate (lower is better) ---
fig, ax = plt.subplots(figsize=(12, 5))

glm_cer = [results[d]["glm"]["cer"] for d in datasets]
paddle_cer = [results[d]["paddle"]["cer"] for d in datasets]

bars1 = ax.bar(x - width/2, glm_cer, width, label="GLM-OCR", color=COLORS["GLM-OCR"], alpha=0.85)
bars2 = ax.bar(x + width/2, paddle_cer, width, label="PaddleOCR-VL-1.5", color=COLORS["PaddleOCR-VL-1.5"], alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(xtick_labels, fontsize=10)
ax.set_ylabel("Character Error Rate (lower = better)", fontsize=12)
ax.set_title("Character Error Rate by Dataset", fontsize=14, fontweight="bold")
ax.legend(fontsize=11)
ax.grid(axis="y", alpha=0.3)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig("plot_cer.png", dpi=150)
plt.show()

In [ ]:
# --- Plot 3: Radar / Spider chart — overall profile ---
# Use (1 - NED) as a similarity score so higher = better for all axes
glm_scores = [results[d]["glm"]["exact_match"] for d in datasets]
paddle_scores = [results[d]["paddle"]["exact_match"] for d in datasets]
short_labels = [DATASET_LABELS[d].replace("\n", " ") for d in datasets]

angles = np.linspace(0, 2 * np.pi, len(datasets), endpoint=False).tolist()
glm_scores += glm_scores[:1]
paddle_scores += paddle_scores[:1]
angles += angles[:1]

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))

ax.plot(angles, glm_scores, "o-", linewidth=2, color=COLORS["GLM-OCR"], label="GLM-OCR")
ax.fill(angles, glm_scores, alpha=0.15, color=COLORS["GLM-OCR"])
ax.plot(angles, paddle_scores, "s-", linewidth=2, color=COLORS["PaddleOCR-VL-1.5"], label="PaddleOCR-VL-1.5")
ax.fill(angles, paddle_scores, alpha=0.15, color=COLORS["PaddleOCR-VL-1.5"])

ax.set_thetagrids(np.degrees(angles[:-1]), short_labels, fontsize=9)
ax.set_ylim(0, 1)
ax.set_yticks([0.25, 0.5, 0.75, 1.0])
ax.set_yticklabels(["25%", "50%", "75%", "100%"], fontsize=7)
ax.set_title("Exact Match Accuracy — Radar Overview", fontsize=13, fontweight="bold", pad=20)
ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1), fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("plot_radar.png", dpi=150)
plt.show()

In [ ]:
# --- Plot 4: Summary — average across all datasets ---
fig, axes = plt.subplots(1, 3, figsize=(12, 4))

metrics_info = [
    ("exact_match", "Avg. Exact Match", True),
    ("cer",         "Avg. CER",         False),
    ("ned",         "Avg. NED",         False),
]

model_keys = [("GLM-OCR", "glm"), ("PaddleOCR-VL-1.5", "paddle")]

for ax, (metric_key, metric_label, higher_better) in zip(axes, metrics_info):
    vals = [
        np.mean([results[d][mk][metric_key] for d in datasets])
        for _, mk in model_keys
    ]
    model_names = [mn for mn, _ in model_keys]
    colors = [COLORS[mn] for mn in model_names]
    bars = ax.bar(model_names, vals, color=colors, alpha=0.85, width=0.4)
    ax.set_title(metric_label, fontsize=12, fontweight="bold")
    ax.set_ylim(0, max(vals) * 1.3 + 0.01)
    note = "↑ higher better" if higher_better else "↓ lower better"
    ax.set_xlabel(note, fontsize=9, color="grey")
    ax.bar_label(bars, fmt="%.3f", padding=3, fontsize=10)
    ax.grid(axis="y", alpha=0.3)
    ax.spines[["top", "right"]].set_visible(False)
    ax.tick_params(axis="x", labelsize=10)

fig.suptitle("Overall Average Performance (all 8 datasets)", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("plot_summary.png", dpi=150, bbox_inches="tight")
plt.show()

## 10. Per-Dataset Qualitative Sample Preview

Show a few sample predictions side by side for manual inspection.

In [ ]:
import math

N_PREVIEW = 3  # samples per dataset to show

for dataset_key in datasets:
    res = results[dataset_key]
    n = min(N_PREVIEW, len(res["labels"]))

    fig, axes = plt.subplots(n, 1, figsize=(10, 3.5 * n), squeeze=False)
    fig.suptitle(
        DATASET_LABELS.get(dataset_key, dataset_key).replace("\n", " "),
        fontsize=14, fontweight="bold", y=1.01
    )

    for i in range(n):
        ax = axes[i][0]
        gt = res["labels"][i]
        g  = res["glm"]["predictions"][i]
        p  = res["paddle"]["predictions"][i]

        img_path = res.get("image_paths", [None] * n)[i]
        if img_path:
            ax.imshow(Image.open(img_path).convert("RGB"))
        ax.axis("off")

        g_em = "✓" if normalise(g) == normalise(str(gt)) else "✗"
        p_em = "✓" if normalise(p) == normalise(str(gt)) else "✗"
        caption = (
            f"Sample {i+1}  |  GT: {gt}\n"
            f"GLM-OCR: {g_em} {g}\n"
            f"PaddleOCR-VL-1.5: {p_em} {p}"
        )
        ax.set_xlabel(caption, fontsize=9, ha="left", x=0, labelpad=6)

    plt.tight_layout()
    plt.savefig(f"preview_{dataset_key}.png", dpi=120, bbox_inches="tight")
    plt.show()


## 11. SmolVLM2 Jersey Number Comparison

[SmolVLM2](https://huggingface.co/HuggingFaceTB/SmolVLM2-500M-Video-Instruct) is a 0.5B vision-language model fine-tuned specifically on jersey number crops (`basketball-jersey-numbers-ocr/3` on Roboflow Universe). Unlike GLM-OCR and PaddleOCR-VL-1.5 which are **general-purpose OCR models**, SmolVLM2 here is **task-specific** — trained on the same type of images we're testing.

This makes the comparison interesting: can a tiny fine-tuned specialist beat two larger generalists?

**Setup required:**
- `ROBOFLOW_API_KEY` — already in secrets (used to download fine-tuned weights)
- `HF_TOKEN` — free Hugging Face token (add to Colab secrets); needed to pull the SmolVLM2 base weights

Inference runs **locally on Colab's GPU** via the open-source `inference` library.

In [ ]:
%%capture
!pip install inference-gpu -q  # use 'inference' instead if no GPU available

In [ ]:
from inference import get_model

SMOLVLM2_MODEL_ID = "basketball-jersey-numbers-ocr/3"
SMOLVLM2_PROMPT = "Read the number."  # prompt from the reference basketball notebook

smolvlm2_model = get_model(model_id=SMOLVLM2_MODEL_ID)
print("SmolVLM2 (fine-tuned) loaded.")

In [ ]:
# Re-load jersey samples (Roboflow caches the download, so this is fast on re-run)
jersey_samples = load_dataset("jersey", n=N_SAMPLES)
jersey_labels  = results["jersey"]["labels"]  # ground truth from the main benchmark run

smolvlm2_preds = []
for sample in tqdm(jersey_samples, desc="SmolVLM2 jersey"):
    image = Image.open(sample["image_path"]).convert("RGB")
    # inference library accepts PIL images; returns list of predictions
    pred = smolvlm2_model.predict(image, SMOLVLM2_PROMPT)[0]
    smolvlm2_preds.append(str(pred).strip())

smolvlm2_metrics = evaluate(smolvlm2_preds, jersey_labels)
results["jersey"]["smolvlm2"] = {**smolvlm2_metrics, "predictions": smolvlm2_preds}

print(f"  GLM-OCR          | EM={results['jersey']['glm']['exact_match']:.3f}  "
      f"CER={results['jersey']['glm']['cer']:.3f}  NED={results['jersey']['glm']['ned']:.3f}")
print(f"  PaddleOCR-VL-1.5 | EM={results['jersey']['paddle']['exact_match']:.3f}  "
      f"CER={results['jersey']['paddle']['cer']:.3f}  NED={results['jersey']['paddle']['ned']:.3f}")
print(f"  SmolVLM2 (ft)    | EM={smolvlm2_metrics['exact_match']:.3f}  "
      f"CER={smolvlm2_metrics['cer']:.3f}  NED={smolvlm2_metrics['ned']:.3f}")

In [ ]:
# --- 3-model jersey comparison: generalist vs fine-tuned specialist ---
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

model_entries = [
    ("GLM-OCR",               "glm",       "#4C72B0"),
    ("PaddleOCR-VL-1.5",      "paddle",    "#DD8452"),
    ("SmolVLM2\n(fine-tuned)", "smolvlm2", "#55A868"),
]
metrics_info = [
    ("exact_match", "Exact Match ↑", True),
    ("cer",         "CER ↓",         False),
    ("ned",         "NED ↓",         False),
]

for ax, (metric_key, metric_label, higher_better) in zip(axes, metrics_info):
    vals   = [results["jersey"][mk][metric_key] for _, mk, _ in model_entries]
    names  = [mn for mn, _, _ in model_entries]
    colors = [c  for _, _, c  in model_entries]
    bars = ax.bar(names, vals, color=colors, alpha=0.85, width=0.5)
    ax.set_title(metric_label, fontsize=12, fontweight="bold")
    ax.set_ylim(0, max(vals) * 1.35 + 0.01)
    ax.bar_label(bars, fmt="%.3f", padding=3, fontsize=10)
    ax.grid(axis="y", alpha=0.3)
    ax.spines[["top", "right"]].set_visible(False)
    ax.tick_params(axis="x", labelsize=9)

fig.suptitle("Jersey Number Recognition — Generalist vs Fine-tuned (3-Model)",
             fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("plot_jersey_3model.png", dpi=150, bbox_inches="tight")
plt.show()

## 12. Final Summary

Two consolidated summary plots:
1. **GLM-OCR vs PaddleOCR-VL-1.5** — Exact Match across all 8 datasets, with per-metric averages
2. **Jersey Number: 3-model showdown** — generalist models vs the fine-tuned SmolVLM2 specialist

In [ ]:
# ── Summary Plot 1: GLM-OCR vs PaddleOCR-VL-1.5 across all 8 datasets ──────

ds_keys   = list(results.keys())
ds_labels = [DATASET_LABELS[d] for d in ds_keys]
n_ds      = len(ds_keys)

glm_em    = [results[d]["glm"]["exact_match"]    for d in ds_keys]
paddle_em = [results[d]["paddle"]["exact_match"] for d in ds_keys]
glm_cer    = [results[d]["glm"]["cer"]    for d in ds_keys]
paddle_cer = [results[d]["paddle"]["cer"] for d in ds_keys]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle("GLM-OCR vs PaddleOCR-VL-1.5 — Full Benchmark Summary",
             fontsize=14, fontweight="bold", y=1.02)

x     = np.arange(n_ds)
width = 0.38

# ── Left: Exact Match ────────────────────────────────────────────────────────
ax = axes[0]
b1 = ax.bar(x - width/2, glm_em,    width, label="GLM-OCR",          color=COLORS["GLM-OCR"],          zorder=3)
b2 = ax.bar(x + width/2, paddle_em, width, label="PaddleOCR-VL-1.5", color=COLORS["PaddleOCR-VL-1.5"], zorder=3)

# Averages as horizontal dashed lines
avg_glm    = sum(glm_em)    / len(glm_em)
avg_paddle = sum(paddle_em) / len(paddle_em)
ax.axhline(avg_glm,    color=COLORS["GLM-OCR"],          linestyle="--", linewidth=1.4,
           label=f"GLM avg {avg_glm:.2f}")
ax.axhline(avg_paddle, color=COLORS["PaddleOCR-VL-1.5"], linestyle="--", linewidth=1.4,
           label=f"Paddle avg {avg_paddle:.2f}")

ax.set_xticks(x)
ax.set_xticklabels(ds_labels, fontsize=8)
ax.set_ylabel("Exact Match Accuracy")
ax.set_ylim(0, 1.12)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.set_title("Exact Match (↑ better)", fontweight="bold")
ax.legend(fontsize=8, ncol=2)
ax.grid(axis="y", alpha=0.3, zorder=0)

# Value labels
for bar in b1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f"{bar.get_height():.2f}", ha="center", va="bottom", fontsize=7)
for bar in b2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f"{bar.get_height():.2f}", ha="center", va="bottom", fontsize=7)

# ── Right: Avg per-metric scorecard ──────────────────────────────────────────
ax2 = axes[1]
metrics_names = ["Exact Match ↑", "CER ↓", "NED ↓"]

def avg(key, model):
    return sum(results[d][model][key] for d in ds_keys) / n_ds

glm_avgs    = [avg("exact_match","glm"),    avg("cer","glm"),    avg("ned","glm")]
paddle_avgs = [avg("exact_match","paddle"), avg("cer","paddle"), avg("ned","paddle")]

xi = np.arange(3)
w  = 0.35
ax2.bar(xi - w/2, glm_avgs,    w, color=COLORS["GLM-OCR"],          label="GLM-OCR",          zorder=3)
ax2.bar(xi + w/2, paddle_avgs, w, color=COLORS["PaddleOCR-VL-1.5"], label="PaddleOCR-VL-1.5", zorder=3)
ax2.set_xticks(xi)
ax2.set_xticklabels(metrics_names, fontsize=10)
ax2.set_ylabel("Score (averaged across 8 datasets)")
ax2.set_ylim(0, 1.1)
ax2.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax2.set_title("Per-Metric Averages Across All Datasets", fontweight="bold")
ax2.legend(fontsize=9)
ax2.grid(axis="y", alpha=0.3, zorder=0)

for bars in [ax2.containers[0], ax2.containers[1]]:
    for bar in bars:
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f"{bar.get_height():.3f}", ha="center", va="bottom", fontsize=9, fontweight="bold")

plt.tight_layout()
plt.savefig("summary_glm_vs_paddle.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Overall winner by Exact Match: "
      f"{'GLM-OCR' if avg_glm > avg_paddle else 'PaddleOCR-VL-1.5'} "
      f"({max(avg_glm, avg_paddle):.3f} vs {min(avg_glm, avg_paddle):.3f})")

In [ ]:
# ── Summary Plot 2: Jersey Number — 3-model showdown ────────────────────────

jersey_entries = [
    ("GLM-OCR",                results["jersey"]["glm"],       COLORS["GLM-OCR"]),
    ("PaddleOCR-VL-1.5",       results["jersey"]["paddle"],    COLORS["PaddleOCR-VL-1.5"]),
    ("SmolVLM2\n(fine-tuned)", results["jersey"]["smolvlm2"], "#55A868"),
]
metric_keys   = ["exact_match", "cer",  "ned"]
metric_labels = ["Exact Match ↑", "CER ↓", "NED ↓"]

fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))
fig.suptitle("Jersey Number Recognition — Generalist vs Specialist",
             fontsize=14, fontweight="bold", y=1.02)

for ax, mkey, mlabel in zip(axes, metric_keys, metric_labels):
    names  = [e[0] for e in jersey_entries]
    values = [e[1][mkey] for e in jersey_entries]
    colors = [e[2] for e in jersey_entries]

    bars = ax.bar(names, values, color=colors, width=0.5, zorder=3)
    ax.set_title(mlabel, fontweight="bold", fontsize=11)
    ax.set_ylim(0, 1.15)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
    ax.grid(axis="y", alpha=0.3, zorder=0)
    ax.tick_params(axis="x", labelsize=9)

    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f"{val:.3f}", ha="center", va="bottom", fontsize=11, fontweight="bold")

    # Highlight the best bar
    best_idx = (values.index(max(values)) if mkey == "exact_match"
                else values.index(min(values)))
    bars[best_idx].set_edgecolor("black")
    bars[best_idx].set_linewidth(2.5)

plt.tight_layout()
plt.savefig("summary_jersey_3models.png", dpi=150, bbox_inches="tight")
plt.show()

# Print scorecard
print("\nJersey Number Scorecard:")
print(f"{'Model':<25} {'EM':>6} {'CER':>6} {'NED':>6}")
print("-" * 45)
for name, res, _ in jersey_entries:
    print(f"{name.replace(chr(10), ' '):<25} {res['exact_match']:>6.3f} {res['cer']:>6.3f} {res['ned']:>6.3f}")